In [1]:
pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 5.8 MB/s  0:00:04 eta 0:00:010:00:01
Note: you may need to restart the kernel to use updated packages.


In [28]:
import pymupdf
import re
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [39]:
doc = pymupdf.open('/Users/yogesh/ML_work/resume_scaner/YogeshJmabhaleCV.pdf')
print(doc.metadata)

page  = doc[0]

resume  = page.get_text()

# print(text)

jd = """About the job
Job Title: MIS Executive / MIS Analyst

Location: Mumbai

Experience: 5 to 7 years

Notice Period: 15 to 30 Days

Job Description:

We are looking for a detail-oriented and analytical MIS Executive/Analyst with 5–7 years of experience to join our team in Bhiwandi, Mumbai. The ideal candidate will be responsible for managing Management Information Systems (MIS), generating accurate and insightful reports and dashboards, and driving data-based decision-making across departments.

Key Responsibilities:

Develop, maintain, and automate daily, weekly, and monthly MIS reports and dashboards for various departments. 
Analyze data to identify trends, variances, and performance metrics to support business decision-making. 
Collaborate with cross-functional teams to gather data requirements and deliver customized reports. 
Use Advanced Excel tools such as Pivot Tables, VLOOKUP, HLOOKUP, INDEX-MATCH, Macros, and Data Validation to process and present data. 
Monitor data integrity and ensure the accuracy of reports and dashboards. 
Prepare business presentations and summary reports for management reviews. 
Support in automation of regular reports using Excel VBA/macros or business intelligence tools (preferred). 
Maintain documentation of report formats, data sources, and report generation processes. 

Key Skills Required:

Proven experience (5–7 years) in MIS reporting, data analysis, and dashboard creation. 
Advanced Excel skills (Pivot Tables, Lookup Functions, Conditional Formatting, Macros, Charts, etc.) 
Good understanding of data analysis and visualization. 
Strong attention to detail and data accuracy. 
Ability to manage large datasets and draw meaningful insights. 
Good communication and stakeholder management skills. 

Preferred Qualifications:

Bachelor’s degree in Commerce, Business Administration, IT, or a related field. 
Experience with Excel VBA/Macros is a plus. 
Experience working in manufacturing, logistics, or warehousing sectors is an advantage. 

Work Location:

On-site – Mumbai 
Candidates from nearby locations preferred"""

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n',' ',text)
    text = re.sub(r'[^a-z0-9\s]','',text)
    text = re.sub(r'\s+',' ',text).strip()
    return  text


clean_resume = clean_text(resume)
clean_jd = clean_text(jd)



{'format': 'PDF 1.4', 'title': 'YogeshJmabhaleCV', 'author': 'Yogesh Jambhale', 'subject': '', 'keywords': 'DAFkwrkWpcI,BAFkwqneO3o,0', 'creator': 'Canva', 'producer': 'Canva', 'creationDate': "D:20260208171236+00'00'", 'modDate': "D:20260208171235+00'00'", 'trapped': '', 'encryption': None}


In [47]:
import spacy
from sentence_transformers import SentenceTransformer, util

In [48]:
print("Loading NLP and Deep Learning Models...")
nlp = spacy.load("en_core_web_md")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Models loaded successfully!\n")

Loading NLP and Deep Learning Models...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Models loaded successfully!



In [49]:
def extract_skills_and_entities(text):
    doc = nlp(text)
    extracted_terms = set()
    
    for ent in doc.ents:
        if ent.label_ in ["ORG", "PRODUCT"]:
            extracted_terms.add(ent.text)
            
    for chunk in doc.noun_chunks:
        clean_chunk = " ".join([token.text for token in chunk if not token.is_stop and not token.is_punct])
        if len(clean_chunk) > 2:
            extracted_terms.add(clean_chunk)
            
    return list(extracted_terms)

In [50]:
def get_missing_skills_by_meaning(resume_skills, jd_skills, similarity_threshold=0.6):
    # Safety check in case the extractor found nothing
    if not jd_skills: return []
    if not resume_skills: return jd_skills
    
    missing_skills = []
    # Convert all resume skills to math vectors at once
    resume_embeddings = sbert_model.encode(resume_skills, convert_to_tensor=True)
    
    for jd_skill in jd_skills:
        # Check each JD skill against the resume
        jd_embedding = sbert_model.encode(jd_skill, convert_to_tensor=True)
        cosine_scores = util.cos_sim(jd_embedding, resume_embeddings)[0]
        
        # Grab the highest percentage match
        best_match_score = cosine_scores.max().item()
        
        # If the best match is below 60%, they are missing the skill!
        if best_match_score < similarity_threshold:
            missing_skills.append(jd_skill)
            
    return missing_skills

In [51]:
def analyze_resume_against_jd(raw_resume, raw_jd):
    # Get clean lists
    resume_skills = extract_skills_and_entities(raw_resume)
    jd_skills = extract_skills_and_entities(raw_jd)
    
    # Find what's actually missing based on meaning
    missing_skills = get_missing_skills_by_meaning(resume_skills, jd_skills)
    
    return missing_skills

In [55]:
print("Analyzing Resume against Job Description...")
final_missing_skills = analyze_resume_against_jd(resume, jd)

Analyzing Resume against Job Description...


In [56]:
print("\n🚨 Actually Missing Skills:")
for skill in final_missing_skills:
    print(f"- {skill}")


🚨 Actually Missing Skills:
- HLOOKUP
- accurate insightful reports
- experience
- Excel VBA/Macros
- 5 7 years 

 Notice Period
- 15 30 Days 

 Job Description
- business intelligence tools
- Preferred Qualifications
- business decision making
- Commerce, Business Administration
- generation processes
- data requirements
- Good understanding
- Commerce Business Administration
- advantage
- Monitor data integrity
- Ability
- related field
- Key Skills Required
- Good communication
- variances
- Lookup Functions
- Charts
- VLOOKUP
- Work Location
- report
- trends
- regular reports
- job 
 Job Title
- cross functional teams
- ideal candidate
- 5–7 years
- summary reports
- large datasets
- detail oriented analytical MIS Executive Analyst
- Bhiwandi
- Conditional Formatting
- dashboard creation
- Strong attention
- report formats
- nearby locations
- Support
- Data Validation
- Pivot Tables
- accuracy
- stakeholder management skills
- data based decision making
- INDEX MATCH
- manufactur